#การเตรียมข้อมูล ตั้งแต่ไฟล์ Excel ดิบจาก CCIB จนได้ข้อมูลที่สะอาดพร้อมวิเคราะห์

#ขั้นตอนที่ 1 — ติดตั้ง Library

In [ ]:
!pip install pandas openpyxl pyarrow scikit-learn

ติดตั้ง library ที่จำเป็น 4 ตัว ได้แก่ pandas สำหรับจัดการตาราง, openpyxl สำหรับอ่านไฟล์ Excel, pyarrow สำหรับบันทึกไฟล์ประสิทธิภาพสูง และ scikit-learn สำหรับทำ Clustering

#ขั้นตอนที่ 2 — รัน Pipeline


Pipeline :
- รัน K-Means กับข้อมูลทั้งหมด 757,038 แถว
- RAM ไม่พอ → Error

--raw-dir = โฟลเดอร์ที่เก็บไฟล์ Excel ดิบจาก CCIB

--output-dir = โฟลเดอร์ที่ pipeline จะสร้าง output ไว้ 3 โฟลเดอร์ย่อย คือ /clean/, /rejected/, /report/

ไม่ใส่ --run-clustering เพราะข้อมูล 757,038 แถว ทำให้ RAM หมด หาก K-Means รันกับข้อมูลทั้งหมด

รันโดยไม่ใส่ --run-clustering ก่อน

In [ ]:
!python "/content/drive/MyDrive/term4/IS/ccib_pipeline.py" \
    --raw-dir "/content/drive/MyDrive/term4/CCIB DATA" \
    --output-dir "/content/drive/MyDrive/term4/IS/output2"

2026-06-05 03:43:36 | INFO | ================================================================================
2026-06-05 03:43:36 | INFO | เริ่มต้น CCIB DATA Pipeline
2026-06-05 03:43:36 | INFO | RAW_DIR    = /content/drive/MyDrive/term4/CCIB DATA
2026-06-05 03:43:36 | INFO | OUTPUT_DIR = /content/drive/MyDrive/term4/IS/output2
2026-06-05 03:43:36 | INFO | ================================================================================
2026-06-05 03:43:37 | INFO | พบไฟล์ Excel ทั้งหมด: 360 ไฟล์
2026-06-05 03:43:39 | INFO | [1/360] อ่านสำเร็จ: สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ่มอายุ เดือนกรกฎาคม 2565.xlsx
2026-06-05 03:43:40 | INFO | [2/360] อ่านสำเร็จ: สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ่มอายุ เดือนกรกฎาคม 2566.xlsx
2026-06-05 03:43:41 | INFO | [3/360] อ่านสำเร็จ: สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ่มอายุ เดือนกรกฎาคม 2567.xlsx
2026-06-05 03:43:43 | INFO | [4/360] อ่านสำเร็จ: สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ่มอายุ เดือนกันยายน 2565.xlsx
2026-06-05 03:43:44 | INFO | [5/360]

ทำให้ข้อมูลเหลือเเค่ปี2565-2566

#ขั้นตอนที่ 3 ตรวจสอบผลลัพธ์

วน loop แสดงชื่อไฟล์และขนาดใน 3 โฟลเดอร์ เพื่อดูว่า pipeline ทำงานสำเร็จหรือไม่

In [ ]:
import os

OUTPUT_DIR = '/content/drive/MyDrive/term4/IS/output'

print("=== output/clean/ ===")
for f in sorted(os.listdir(OUTPUT_DIR + '/clean')):
    size = os.path.getsize(OUTPUT_DIR + '/clean/' + f)
    print(f"  {f}  ({size/1024:.0f} KB)")

print("\n=== output/rejected/ ===")
for f in sorted(os.listdir(OUTPUT_DIR + '/rejected')):
    print(f"  {f}")

print("\n=== output/report/ ===")
for f in sorted(os.listdir(OUTPUT_DIR + '/report')):
    print(f"  {f}")

=== output/clean/ ===
  ccib_all_clean.csv  (397554 KB)
  ccib_all_clean.parquet  (1778 KB)
  ccib_cluster_base.csv  (160683 KB)
  ccib_cluster_base.parquet  (1715 KB)
  ccib_main_clean.csv  (393472 KB)
  ccib_main_clean.parquet  (1738 KB)
  viz_age_summary.csv  (0 KB)
  viz_age_summary.parquet  (5 KB)
  viz_case_type_by_age.csv  (24 KB)
  viz_case_type_by_age.parquet  (9 KB)
  viz_case_type_by_gender.csv  (11 KB)
  viz_case_type_by_gender.parquet  (7 KB)
  viz_case_type_summary.csv  (5 KB)
  viz_case_type_summary.parquet  (6 KB)
  viz_gender_summary.csv  (0 KB)
  viz_gender_summary.parquet  (4 KB)
  viz_job_summary.csv  (1 KB)
  viz_job_summary.parquet  (5 KB)
  viz_monthly_trend.csv  (1 KB)
  viz_monthly_trend.parquet  (5 KB)
  viz_province_summary.csv  (3 KB)
  viz_province_summary.parquet  (6 KB)

=== output/rejected/ ===
  01_rejected_files.csv
  02_rejected_rows.csv

=== output/report/ ===
  01_file_summary.csv
  02_column_quality_report.csv
  02_column_quality_report.gsheet
  03

In [ ]:
import pandas as pd

OUTPUT_DIR = '/content/drive/MyDrive/term4/IS/output'

# 1. เช็คไฟล์ที่ถูก reject
print("=== ไฟล์ที่ถูก Reject ===")
df_rejected = pd.read_csv(OUTPUT_DIR + '/rejected/01_rejected_files.csv')
print(f"จำนวนไฟล์ที่ reject: {len(df_rejected)}")
print(df_rejected[['source_file','rejected_reason']].to_string(index=False))

# 2. เช็ค Data Quality Summary
print("\n=== Data Quality Summary ===")
df_summary = pd.read_csv(OUTPUT_DIR + '/report/07_data_quality_summary.csv')
print(df_summary.T.to_string())

# 3. เช็ค viz_age_summary ที่ขนาด 0 KB
print("\n=== viz_age_summary (0 KB) ===")
df_age = pd.read_csv(OUTPUT_DIR + '/clean/viz_age_summary.csv')
print(f"Shape: {df_age.shape}")
print(df_age.head())

=== ไฟล์ที่ถูก Reject ===
จำนวนไฟล์ที่ reject: 10
                                                                   source_file                                     rejected_reason
                      สถิติคดีออนไลน์แบ่งตามหน่วยงาน เดือนสิงหาคม ปี 2567.xlsx                duplicate_2567_same_as_previous_year
                                สถิติคดีแบ่งตามอาชีพ เดือนกรกฎาคม ปี 2566.xlsx                                       year_mismatch
สถิติจำนวนคดีที่ทางสถานีตำรวจได้รับแบ่งตามกลุ่มอายุ เดือน สิงหาคม ปี 2567.xlsx                duplicate_2567_same_as_previous_year
                                สถิติจำนวนคดีออนไลน์ เดือนกุมภาพันธ์ 2567.xlsx                                       year_mismatch
                                   สถิติจำนวนคดีออนไลน์ เดือนพฤษภาคม 2567.xlsx duplicate_2567_same_as_previous_year; year_mismatch
                                    สถิติจำนวนคดีออนไลน์ เดือนมกราคม 2567.xlsx duplicate_2567_same_as_previous_year; year_mismatch
                                 

สิ่งที่ดีแล้ว ✓
Reject ถูกต้องครบ — Pipeline ตรวจพบไฟล์ปัญหาได้อัตโนมัติ 10 ไฟล์ และ viz_age_summary.csv ที่ขนาด 0 KB จริง ๆ มีข้อมูลครบ 8 กลุ่มอายุ แค่ไฟล์มีขนาดเล็กมาก

Pipeline ตรวจพบและ reject ไฟล์ที่มีปัญหาอัตโนมัติ 10 ไฟล์ ด้วยเหตุผล year_mismatch หมายความว่าชื่อไฟล์บอกว่าเป็นปี 2566 แต่ข้อมูลข้างในเป็นปี 2565 เพราะ CCIB อัปโหลดไฟล์ผิด
สรุปผลที่ได้:

ข้อมูลที่ผ่านการทำความสะอาดอยู่ในช่วงมีนาคม 2565 ถึงสิงหาคม 2567
ปี 2567 ที่อยู่ใน clean data คือ กรกฎาคม และ สิงหาคม 2567 ซึ่งข้อมูลถูกต้องจริง

In [ ]:
import pandas as pd

OUTPUT_DIR = '/content/drive/MyDrive/term4/IS/output'

# ดูชื่อคอลัมน์จริงของ year_check_report
df_year = pd.read_csv(OUTPUT_DIR + '/report/03_year_check_report.csv')
print("คอลัมน์:", list(df_year.columns))
print(f"Shape: {df_year.shape}")
print("\nตัวอย่าง 3 แถวแรก:")
print(df_year.head(3).to_string())

# เช็คช่วงเวลาที่ผ่าน clean
print("\n=== ช่วงเวลาที่ผ่าน Clean ===")
df_coverage = pd.read_csv(OUTPUT_DIR + '/report/06_monthly_coverage_clean.csv')
print("คอลัมน์:", list(df_coverage.columns))
print(df_coverage.to_string(index=False))

คอลัมน์: ['source_category', 'source_file', 'source_path', 'month_from_file', 'year_from_file', 'years_in_data', 'year_check_status', 'year_mismatch']
Shape: (360, 8)

ตัวอย่าง 3 แถวแรก:
                                 source_category                                                         source_file                                                                                                                                              source_path  month_from_file  year_from_file  years_in_data             year_check_status  year_mismatch
0  สถิติคดีออนไลน์เเบ่งตามประเภทคดีเเละกลุ่มอายุ  สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ่มอายุ เดือนกรกฎาคม 2565.xlsx  /content/drive/MyDrive/term4/CCIB DATA/สถิติคดีออนไลน์เเบ่งตามประเภทคดีเเละกลุ่มอายุ/สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ่มอายุ เดือนกรกฎาคม 2565.xlsx                7            2565            NaN  no_year_column_or_empty_year          False
1  สถิติคดีออนไลน์เเบ่งตามประเภทคดีเเละกลุ่มอายุ  สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ

 รายงานผลการตรวจสอบปี ที่ Pipeline สร้างขึ้นอัตโนมัติ
สิ่งที่เห็นในตัวอย่าง 3 แถวนี้คือ Pipeline กำลังรายงานว่าไฟล์ "สถิติคดีออนไลน์แบ่งตามประเภทคดีและกลุ่มอายุ" ทั้ง 3 เดือน (กรกฎาคม 2565, 2566, 2567) มีสถานะ no_year_column_or_empty_year หมายความว่าไฟล์เหล่านี้ไม่มีคอลัมน์ "ปี" ในเนื้อหา จึงไม่สามารถเปรียบเทียบปีในชื่อไฟล์กับปีในเนื้อหาได้ แต่ year_mismatch = False คือ ไม่ถูก reject เพราะถือว่าไม่มีหลักฐานพอที่จะตัดออก

In [ ]:
#ดูตัวอย่างข้อมูลในccib_main_clean.csv

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/term4/IS/output/clean/ccib_main_clean.csv',
                 encoding='utf-8-sig')

print(df.shape)
print(df.head(5).to_string())

(530908, 28)
                                ประเภทคดี กลุ่มอายุ  จำนวนคดี  จำนวนคดี (เพศชาย)  จำนวนคดี (เพศหญิง)  จำนวนคดี (ไม่ระบุเพศ)                                  source_file       source_category                                                                                              source_path source_sheet  month_from_file  year_from_file case_type_name หน่วยงาน    ปี    เดือน  จำนวนผู้เสียหาย                  อาชีพ   เพศ        จังหวัด    อำเภอ             ตำบล ประเภทความเสียหาย  มูลค่าความเสียหาย  year_in_data  is_rejected  rejected_reason  เดือนเลข
0  15.คดีอาชญากรรมทางเทคโนโลยีลักษณะอื่นๆ  30-44 ปี         1                NaN                 NaN                    NaN  สถิติจำนวนคดีออนไลน์ เดือนกรกฎาคม 2565.xlsx  สถิติจำนวนคดีออนไลน์  /content/drive/MyDrive/term4/CCIB DATA/สถิติจำนวนคดีออนไลน์/สถิติจำนวนคดีออนไลน์ เดือนกรกฎาคม 2565.xlsx       Sheet1                7            2565        ไม่ระบุ  บช.สอท.  2565  กรกฎาคม              1.0          ธุรกิจส่วนตัว  หญิง  

In [ ]:
#ดูตัวอย่างข้อมูลในccib_cluster_base.csv

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/term4/IS/output/clean/ccib_cluster_base.csv',
                 encoding='utf-8-sig')

print(df.shape)
print(df.head(5).to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(529959, 15)
     ปี  เดือนเลข จังหวัด     อำเภอ           ตำบล หน่วยงาน                               ประเภทคดี ประเภทความเสียหาย   เพศ กลุ่มอายุ                  อาชีพ  จำนวนคดี  จำนวนผู้เสียหาย  มูลค่าความเสียหาย  record_count
0  2565         3  กระบี่  คลองท่อม  คลองท่อมเหนือ      ภ.8  15.คดีอาชญากรรมทางเทคโนโลยีลักษณะอื่นๆ              เงิน  หญิง  45-59 ปี                เกษตรกร         2              2.0                0.0             1
1  2565         3  กระบี่  คลองท่อม  คลองท่อมเหนือ      ภ.8  15.คดีอาชญากรรมทางเทคโนโลยีลักษณะอื่นๆ           ไม่ระบุ  หญิง  45-59 ปี                เกษตรกร         2              2.0                0.0             1
2  2565         3  กระบี่  คลองท่อม    คลองท่อมใต้      ภ.8  15.คดีอาชญากรรมทางเทคโนโลยีลักษณะอื่นๆ    ทรัพย์สินอื่นๆ   ชาย  18-21 ปี      นักเรียน/นักศึกษา         1              1.0                0.0     

In [ ]:
import pandas as pd

OUTPUT_DIR = '/content/drive/MyDrive/term4/IS/output'
df_age = pd.read_csv(OUTPUT_DIR + '/clean/viz_age_summary.csv')
print(df_age)
# ดูว่า "ไม่ระบุ" มีจำนวนคดีเท่าไหร่จริงๆ
not_specified = df_age[df_age['กลุ่มอายุ'] == 'ไม่ระบุ']['จำนวนคดี'].sum()
print(f"ไม่ระบุ: {not_specified:,} คดี")

     กลุ่มอายุ  จำนวนคดี  จำนวนผู้เสียหาย  มูลค่าความเสียหาย
0      1-14 ปี      1095           1058.0       9.300000e+07
1     15-17 ปี      7038           6828.0       3.250000e+07
2     18-21 ปี     62386          60302.0       1.975000e+08
3     22-29 ปี    263879         256012.0       6.210000e+08
4     30-44 ปี    419560         407180.0       4.031000e+09
5     45-59 ปี    186924         181520.0       2.931000e+09
6  60 ปีขึ้นไป     53598          52299.0       1.831000e+09
7      ไม่ระบุ    838509         810287.0       1.516400e+10
ไม่ระบุ: 838,509 คดี


ไม่ระบุ: 838,509 คดี-> มาจากการsumคดี

In [ ]:
import pandas as pd

df = pd.read_parquet('/content/drive/MyDrive/term4/IS/output/clean/ccib_main_clean.parquet')

# เช็คว่า "ไม่ระบุ" ในกลุ่มอายุ มาจาก source_category ไหน และ sum จำนวนคดีเป็นเท่าไหร่
result = df[df['กลุ่มอายุ'] == 'ไม่ระบุ'].groupby('source_category').agg(
    แถว=('จำนวนคดี', 'count'),
    sum_คดี=('จำนวนคดี', 'sum')
).reset_index()

print(result)
print(f"\nรวม sum จำนวนคดี ไม่ระบุ ทั้งหมด: {result['sum_คดี'].sum():,.0f}")

                          source_category   แถว  sum_คดี
0                    สถิติจำนวนคดีออนไลน์  1434    19504
1  สถิติมูลค่าความเสียหายแบ่งตามกลุ่มอายุ    30    19455
2  สถิติมูลค่าความเสียหายแบ่งตามประเภทคดี   681   399775
3        สถิติมูลค่าความเสียหายแบ่งตามเพศ   239   399775

รวม sum จำนวนคดี ไม่ระบุ ทั้งหมด: 838,509


ช็คดูว่า ccib_cluster_base ต่างจาก ccib_main_clean ยังไง

In [ ]:
df_main = pd.read_parquet(OUTPUT_DIR + '/clean/ccib_main_clean.parquet')
df_cluster = pd.read_parquet(OUTPUT_DIR + '/clean/ccib_cluster_base.parquet')

print("=== ccib_main_clean ===")
print(f"แถว: {len(df_main):,}")
print(f"source_category:\n{df_main['source_category'].value_counts()}\n")

print("=== ccib_cluster_base ===")
print(f"แถว: {len(df_cluster):,}")
if 'source_category' in df_cluster.columns:
    print(f"source_category:\n{df_cluster['source_category'].value_counts()}")
else:
    print("ไม่มีคอลัมน์ source_category")
    print(f"คอลัมน์ทั้งหมด: {df_cluster.columns.tolist()}")

=== ccib_main_clean ===
แถว: 530,908
source_category:
source_category
สถิติจำนวนคดีออนไลน์                      529481
สถิติมูลค่าความเสียหายแบ่งตามประเภทคดี       681
สถิติมูลค่าความเสียหายแบ่งตามกลุ่มอายุ       507
สถิติมูลค่าความเสียหายแบ่งตามเพศ             239
Name: count, dtype: int64

=== ccib_cluster_base ===
แถว: 529,959
ไม่มีคอลัมน์ source_category
คอลัมน์ทั้งหมด: ['ปี', 'เดือนเลข', 'จังหวัด', 'อำเภอ', 'ตำบล', 'หน่วยงาน', 'ประเภทคดี', 'ประเภทความเสียหาย', 'เพศ', 'กลุ่มอายุ', 'อาชีพ', 'จำนวนคดี', 'จำนวนผู้เสียหาย', 'มูลค่าความเสียหาย', 'record_count']


In [ ]:
# เช็คว่า ccib_cluster_base มีหมวดอะไรบ้าง
# โดยเทียบกับ ccib_main_clean
df_main = pd.read_parquet(OUTPUT_DIR + '/clean/ccib_main_clean.parquet')
df_cluster = pd.read_parquet(OUTPUT_DIR + '/clean/ccib_cluster_base.parquet')

# เช็คว่า ccib_cluster_base กรองยังไง
# ลองดู keyword ที่ pipeline ใช้กรอง
print("=== main_category_keyword ใน pipeline ===")
print("สถิติจำนวนคดีออนไลน์|สถิติมูลค่าความเสียหาย")

# เช็คแถวที่อยู่ใน cluster_base แต่ไม่อยู่ใน main_clean
print(f"\nแถวใน main_clean: {len(df_main):,}")
print(f"แถวใน cluster_base: {len(df_cluster):,}")
print(f"ต่างกัน: {len(df_cluster) - df_main[df_main['source_category'] == 'สถิติจำนวนคดีออนไลน์'].shape[0]:,} แถว")

# ดูว่า cluster_base มีปีอะไรบ้าง
print(f"\nปีใน cluster_base:")
print(df_cluster['ปี'].value_counts().sort_index())

print(f"\nปีใน main_clean (เฉพาะสถิติจำนวนคดีออนไลน์):")
print(df_main[df_main['source_category']=='สถิติจำนวนคดีออนไลน์']['ปี'].value_counts().sort_index())

=== main_category_keyword ใน pipeline ===
สถิติจำนวนคดีออนไลน์|สถิติมูลค่าความเสียหาย

แถวใน main_clean: 530,908
แถวใน cluster_base: 529,959
ต่างกัน: 478 แถว

ปีใน cluster_base:
ปี
2565    254254
2566    275705
Name: count, dtype: int64

ปีใน main_clean (เฉพาะสถิติจำนวนคดีออนไลน์):
ปี
2565    254060
2566    275421
Name: count, dtype: int64


ปี 2565: cluster_base มีมากกว่า 254,254 - 254,060 = 194 แถว

ปี 2566: cluster_base มีมากกว่า 275,705 - 275,421 = 284 แถว

รวม: 478 แถว

แปลว่า ccib_cluster_base ไม่ได้กรองแค่ สถิติจำนวนคดีออนไลน์ อย่างเดียว แต่มีแถวพิเศษเพิ่มมาด้วย ลองเช็คดูค่ะว่ามาจากไหน

In [ ]:
# ดู keyword ที่ pipeline ใช้สร้าง ccib_cluster_base
import subprocess
result = subprocess.run(
    ['grep', '-n', 'cluster_base',
     '/content/drive/MyDrive/term4/IS/ccib_pipeline.py'],
    capture_output=True, text=True
)
print(result.stdout)

604:def build_cluster_base(main_df: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
637:        cluster_base = (
644:        cluster_base = (
649:        cluster_base["record_count"] = (
655:    return cluster_base
695:def run_clustering(cluster_base: pd.DataFrame, output_dir: Path, logger: logging.Logger) -> None:
696:    if cluster_base.empty:
697:        logger.warning("cluster_base ว่าง จึงไม่ทำ clustering")
710:    df = cluster_base.copy()
885:    cluster_base = build_cluster_base(main_df, logger)
886:    if not cluster_base.empty:
887:        write_csv(cluster_base, clean_dir / "ccib_cluster_base.csv")
888:        write_parquet_if_possible(cluster_base, clean_dir / "ccib_cluster_base.parquet", logger)
894:        run_clustering(cluster_base, output_dir, logger)



In [ ]:
result = subprocess.run(
    ['sed', '-n', '604,660p',
     '/content/drive/MyDrive/term4/IS/ccib_pipeline.py'],
    capture_output=True, text=True
)
print(result.stdout)

def build_cluster_base(main_df: pd.DataFrame, logger: logging.Logger) -> pd.DataFrame:
    """
    เตรียมข้อมูลสำหรับ clustering โดย aggregate ตามตัวแปรกลุ่มที่มีจริง
    """
    if main_df.empty:
        return pd.DataFrame()

    group_candidates = [
        "ปี",
        "เดือนเลข",
        "จังหวัด",
        "อำเภอ",
        "ตำบล",
        "หน่วยงาน",
        "ประเภทคดี",
        "ประเภทความเสียหาย",
        "เพศ",
        "กลุ่มอายุ",
        "อาชีพ",
    ]

    group_cols = find_existing_columns(main_df, group_candidates)

    if not group_cols:
        logger.warning("ไม่พบคอลัมน์สำหรับ groupby ใน main dataset")
        return pd.DataFrame()

    # หา metric columns
    metric_candidates = ["จำนวนคดี", "จำนวนผู้เสียหาย", "มูลค่าความเสียหาย"]
    metric_cols = [c for c in metric_candidates if c in main_df.columns and pd.api.types.is_numeric_dtype(main_df[c])]

    if not metric_cols:
        # ถ้าไม่มี metric ให้ใช้จำนวนแถวแทน
        cluster_base = (
            main_df.groupby

ccib_cluster_base รวมข้อมูลมูลค่าความเสียหายเข้าไปด้วยผ่านคอลัมน์ มูลค่าความเสียหาย แต่ไม่กระทบ Risk Score เลยค่ะ เพราะ Phase 4 ใช้แค่คอลัมน์ จำนวนคดี ในการคำนวณ case_rate และ growth_rate อยู่แล้ว
สรุปคือไม่ต้องแก้อะไรค่ะ ใช้ ccib_cluster_base.parquet ใน Phase 4 ต่อไปได้เลย